<h5>Simple Security Incident Response Agent </h5>

The agent will classify the alerts to High, medium and low severity and mark it respectively.

In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
openai = OpenAI()

In [31]:
alerts = []
severity = []
responsed = []

In [ ]:
def get_incident_report() -> str:
    result = ""
    for index, todo in enumerate(alerts):
        if responsed[index]:
            if severity[index] == "HIGH":
                result += f"Alert #{index + 1}: [red][strike]{todo}[/strike][/red] : [red]{severity[index]}[/red]\n" 
            elif severity[index] == "MEDIUM":
                result += f"Alert #{index + 1}: [yellow][strike]{todo}[/strike][/yellow] : [yellow]{severity[index]}[/yellow]\n"
            elif severity[index] == "LOW":
                result += f"Alert #{index + 1}: [green][strike]{todo}[/strike][/green] : [green]{severity[index]}[/green]\n"
        else:
            result += f"Alert #{index + 1}: {todo}\n"
    show(result)
    return result

In [33]:
get_incident_report()

''

In [34]:
def create_alerts(descriptions: list[str]) -> str:
    alerts.extend(descriptions)
    responsed.extend([False] * len(descriptions))
    severity.extend([None] * len(descriptions))
    return get_incident_report()

In [35]:
def mark_response(index: int,  severity_of_alert: str, completion_notes: str) -> str:
    if 1 <= index <= len(alerts):
        responsed[index - 1] = True
        severity[index - 1] = severity_of_alert
    else:
        return "No alert at this index."
    Console().print(completion_notes)
    return get_incident_report()

In [36]:
alerts, responsed, severity = [], [], []

create_alerts(["Failed login", "DoS attack", "Suspicious Login"])

Alert #1: Failed login
Alert #2: DoS attack
Alert #3: Suspicious Login

'Alert #1: Failed login\nAlert #2: DoS attack\nAlert #3: Suspicious Login\n'

In [ ]:
mark_response(1, "LOW", "user logging")
mark_response(2, "HIGH", "stopped by WAF")
mark_response(3, "MEDIUM", "user location changed - verified")

user logging

Alert #1: Failed login : LOW
Alert #2: DoS attack
Alert #3: Suspicious Login

stopped by WAF

Alert #1: Failed login : LOW
Alert #2: DoS attack : HIGH
Alert #3: Suspicious Login

user location changed - verified

Alert #1: Failed login : LOW
Alert #2: DoS attack : HIGH
Alert #3: Suspicious Login : MEDIUM

'Alert #1: [green][strike]Failed login[/strike][/green] : [green]LOW[/green]\nAlert #2: [red][strike]DoS attack[/strike][/red] : [red]HIGH[/red]\nAlert #3: [yellow][strike]Suspicious Login[/strike][/yellow] : [yellow]MEDIUM[/yellow]\n'

In [38]:
create_alerts_json = {
    "name": "create_alerts",
    "description": "Add new alerts from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [39]:
mark_response_json = {
    "name": "mark_response",
    "description": "Mark response the alerts at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'severity_of_alert': {
                'description' : 'give severity as LOW, MEDIUM or HIGH',
                'title': 'Severity Of Alerts',
                'type': 'string'
            },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'severity_of_alert', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [40]:
tools =[{ 'type':'function', 'function': create_alerts_json},
        {'type': 'function', 'function': mark_response_json}]

In [57]:

def handle_tool_calls(tool_calls):
    results= []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else { }
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id })
    return results    
    

In [58]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [50]:
message = "Generate five cybersecurity alerts - one liners. It needs to be very minimal words and do not classify based on severity."
generate_alerts_mssg = [{"role": "user", "content": message}]
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages= generate_alerts_mssg,
)
alert_results = response.choices[0].message.content
print(alert_results)

- Multiple failed SSH logins from 198.51.100.23
- Unusual outbound traffic spike to 203.0.113.45
- Malware detected and quarantined: trojan.exe
- New administrative account created: svc_backup
- Suspicious PowerShell execution on host DC-01


In [59]:
system_message = """
You are given a problem to solve, by using your alert tools to plan a list of steps, then carrying out each step in turn.
Now use the alerts list tools, create a list of alerts, carry out the steps, and reply with the solution to classify alerts.
After each classification mention why are classifying for that severity. 
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = "Classify the alerts accordingly." + alert_results
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [60]:
alerts, responsed, severity = [], [], []
loop(messages)

Alert #1: Multiple failed SSH logins from 198.51.100.23
Alert #2: Unusual outbound traffic spike to 203.0.113.45
Alert #3: Malware detected and quarantined: trojan.exe
Alert #4: New administrative account created: svc_backup
Alert #5: Suspicious PowerShell execution on host DC-01

Severity: MEDIUM
Reason: Repeated failed SSH logins indicate probable brute-force/credential-stuffing attempts. No confirmation of 
successful access, so elevated but not yet critical.

Alert #1: Multiple failed SSH logins from 198.51.100.23 : MEDIUM
Alert #2: Unusual outbound traffic spike to 203.0.113.45
Alert #3: Malware detected and quarantined: trojan.exe
Alert #4: New administrative account created: svc_backup
Alert #5: Suspicious PowerShell execution on host DC-01

Severity: HIGH
Reason: Sudden outbound traffic spikes to a specific external IP can indicate active data exfiltration or C2 
communications, requiring immediate investigation/containment.

Alert #1: Multiple failed SSH logins from 198.51.100.23 : MEDIUM
Alert #2: Unusual outbound traffic spike to 203.0.113.45 : HIGH
Alert #3: Malware detected and quarantined: trojan.exe
Alert #4: New administrative account created: svc_backup
Alert #5: Suspicious PowerShell execution on host DC-01

...

Severity: HIGH
Reason: Creation of a new administrative account is a strong indicator of privilege escalation/persistence. Treat 
as potentially unauthorized until verified.

Alert #1: Multiple failed SSH logins from 198.51.100.23 : MEDIUM
Alert #2: Unusual outbound traffic spike to 203.0.113.45 : HIGH
Alert #3: Malware detected and quarantined: trojan.exe : HIGH
Alert #4: New administrative account created: svc_backup : HIGH
Alert #5: Suspicious PowerShell execution on host DC-01

Severity: HIGH
Reason: Suspicious PowerShell on a Domain Controller (DC-01) is high-risk due to likely post-exploitation activity 
(e.g., recon, credential dumping, lateral movement) impacting the whole domain.

Alert #1: Multiple failed SSH logins from 198.51.100.23 : MEDIUM
Alert #2: Unusual outbound traffic spike to 203.0.113.45 : HIGH
Alert #3: Malware detected and quarantined: trojan.exe : HIGH
Alert #4: New administrative account created: svc_backup : HIGH
Alert #5: Suspicious PowerShell execution on host DC-01 : HIGH

Alert Severity Classification

1) MEDIUM — Multiple failed SSH logins from 198.51.100.23  
   Why: Indicates likely brute-force/credential-stuffing activity, but no confirmed successful login/impact yet.

2) HIGH — Unusual outbound traffic spike to 203.0.113.45  
   Why: Could represent active data exfiltration or command-and-control traffic; requires immediate 
triage/containment.

3) HIGH — Malware detected and quarantined: trojan.exe  
   Why: Confirmed malicious code presence; even quarantined, it implies compromise and potential 
persistence/lateral movement.

4) HIGH — New administrative account created: svc_backup  
   Why: Potential unauthorized privilege escalation/persistence mechanism; high impact if malicious.

5) HIGH — Suspicious PowerShell execution on host DC-01  
   Why: Suspicious scripting on a Domain Controller is high-risk and commonly tied to post-exploitation (credential
theft/lateral movement) with domain-wide impact.